# Load Enrollments Table
## First Load 2015 up to 2020

In [ ]:
import requests
import pandas as pd
import pyodbc
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- Credentials ---
username = "6b3661eb-7413-444c-9ee7-648d0a0915e6"
password = "6b3661eb-7413-444c-9ee7-648d0a0915e6"

# --- SQL connection ---
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=BRW03WFTATHASD;"
    "DATABASE=WFTD_data_warehouse;"
    "Trusted_Connection=yes;"
)

cursor = conn.cursor()
cursor.fast_executemany = True

base_url = "https://api.365.systems/odata/v2/Enrollments/IncludeDeletedUsers()"
table_name = "Enrollments"

top = 5000
first_batch = True

# ✅ YEAR LOOP (2015–2020)
for year in range(2015, 2021):

    start_str = f"{year}-01-01T00:00:00Z"
    end_str = f"{year+1}-01-01T00:00:00Z"

    print(f"\nProcessing YEAR: {year}")

    skip = 0

    while True:

        url = (
            f"{base_url}?"
            f"$filter=RegistrationDate ge {start_str} and RegistrationDate lt {end_str}"
            f"&$top={top}&$skip={skip}"
        )

        response = requests.get(
            url,
            auth=(username, password),
            verify=False,
            timeout=120
        )

        data = response.json().get("value", [])

        if not data:
            break

        final_rows = []

        # ✅ Expand Roles INCLUDING blank roles
        for row in data:

            roles = row.get("Roles")

            # ✅ Case 1: Roles is a list
            if isinstance(roles, list):

                # ✅ Empty list → preserve row with blank role
                if len(roles) == 0:
                    new_row = row.copy()
                    new_row["Role"] = None
                    final_rows.append(new_row)

                else:
                    for r in roles:
                        new_row = row.copy()

                        if isinstance(r, dict):
                            new_row["Role"] = r.get("Name")
                        else:
                            new_row["Role"] = r

                        final_rows.append(new_row)

            # ✅ Case 2: Roles is object
            elif isinstance(roles, dict):
                new_row = row.copy()
                new_row["Role"] = roles.get("Name")
                final_rows.append(new_row)

            # ✅ Case 3: Roles is None/string
            else:
                new_row = row.copy()
                new_row["Role"] = roles
                final_rows.append(new_row)

        df = pd.DataFrame(final_rows)

        # ✅ Remove original Roles column
        if "Roles" in df.columns:
            df = df.drop(columns=["Roles"])

        # ✅ Preserve NULL properly
        df = df.where(pd.notnull(df), None)

        # ✅ Convert safely to string
        for col in df.columns:
            df[col] = df[col].apply(
                lambda x: str(x)[:4000] if x is not None else None
            )

        # ✅ Create table only first time
        if first_batch:

            cols = [f"[{col}] NVARCHAR(MAX)" for col in df.columns]

            cursor.execute(f"""
            IF OBJECT_ID('{table_name}', 'U') IS NULL
            CREATE TABLE {table_name} (
                {', '.join(cols)}
            )
            """)

            conn.commit()
            first_batch = False

        # ✅ Insert
        cols = ",".join([f"[{c}]" for c in df.columns])
        placeholders = ",".join(["?"] * len(df.columns))

        cursor.executemany(
            f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})",
            df.values.tolist()
        )

        conn.commit()

        skip += top

        print(f"Year {year} → Inserted {len(df)} rows | skip={skip}")

print("\n✅ Batch 1 (2015–2020) completed successfully")

# Load Enrollments Table
## Second Load 2021 up to 2026

In [ ]:
import requests
import pandas as pd
import pyodbc
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- Credentials ---
username = "6b3661eb-7413-444c-9ee7-648d0a0915e6"
password = "6b3661eb-7413-444c-9ee7-648d0a0915e6"

# --- SQL connection ---
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=BRW03WFTATHASD;"
    "DATABASE=WFTD_data_warehouse;"
    "Trusted_Connection=yes;"
)

cursor = conn.cursor()
cursor.fast_executemany = True

base_url = "https://api.365.systems/odata/v2/Enrollments/IncludeDeletedUsers()"
table_name = "Enrollments"

top = 5000
first_batch = True

# ✅ YEAR LOOP (2026-2027). Change This for each year.
for year in range(2026,2027):

    start_str = f"{year}-01-01T00:00:00Z"
    end_str = f"{year+1}-01-01T00:00:00Z"

    print(f"\nProcessing YEAR: {year}")

    skip = 0

    while True:

        url = (
            f"{base_url}?"
            f"$filter=RegistrationDate ge {start_str} and RegistrationDate lt {end_str}"
            f"&$top={top}&$skip={skip}"
        )

        response = requests.get(
            url,
            auth=(username, password),
            verify=False,
            timeout=120
        )

        data = response.json().get("value", [])

        if not data:
            break

        final_rows = []

        # ✅ Expand Roles INCLUDING blank roles
        for row in data:

            roles = row.get("Roles")

            # ✅ Case 1: Roles is a list
            if isinstance(roles, list):

                # ✅ Empty list → preserve row with blank role
                if len(roles) == 0:
                    new_row = row.copy()
                    new_row["Role"] = None
                    final_rows.append(new_row)

                else:
                    for r in roles:
                        new_row = row.copy()

                        if isinstance(r, dict):
                            new_row["Role"] = r.get("Name")
                        else:
                            new_row["Role"] = r

                        final_rows.append(new_row)

            # ✅ Case 2: Roles is object
            elif isinstance(roles, dict):
                new_row = row.copy()
                new_row["Role"] = roles.get("Name")
                final_rows.append(new_row)

            # ✅ Case 3: Roles is None/string
            else:
                new_row = row.copy()
                new_row["Role"] = roles
                final_rows.append(new_row)

        df = pd.DataFrame(final_rows)

        # ✅ Remove original Roles column
        if "Roles" in df.columns:
            df = df.drop(columns=["Roles"])

        # ✅ Preserve NULL properly
        df = df.where(pd.notnull(df), None)

        # ✅ Convert safely to string
        for col in df.columns:
            df[col] = df[col].apply(
                lambda x: str(x)[:4000] if x is not None else None
            )

        # ✅ Create table only first time
        if first_batch:

            cols = [f"[{col}] NVARCHAR(MAX)" for col in df.columns]

            cursor.execute(f"""
            IF OBJECT_ID('{table_name}', 'U') IS NULL
            CREATE TABLE {table_name} (
                {', '.join(cols)}
            )
            """)

            conn.commit()
            first_batch = False

        # ✅ Insert
        cols = ",".join([f"[{c}]" for c in df.columns])
        placeholders = ",".join(["?"] * len(df.columns))

        cursor.executemany(
            f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})",
            df.values.tolist()
        )

        conn.commit()

        skip += top

        print(f"Year {year} → Inserted {len(df)} rows | skip={skip}")

print("\n✅ Batch 1 (2026) completed successfully")

# Load 'Courses' Table

In [ ]:
import requests
import pandas as pd
import pyodbc
import urllib3
import warnings

warnings.filterwarnings("ignore")
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- Credentials ---
username = "6b3661eb-7413-444c-9ee7-648d0a0915e6"
password = "6b3661eb-7413-444c-9ee7-648d0a0915e6"

# --- SQL Connection ---
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=BRW03WFTATHASD;"
    "DATABASE=WFTD_data_warehouse;"
    "Trusted_Connection=yes;"
)

cursor = conn.cursor()
cursor.fast_executemany = True

# ✅ Expand Rating + Publishing
base_url = "https://api.365.systems/odata/v2/Courses?$expand=Rating,Publishing"

table_name = "Courses"

top = 5000
skip = 0
first_batch = True

while True:
    url = f"{base_url}&$top={top}&$skip={skip}"

    try:
        response = requests.get(
            url,
            auth=(username, password),
            verify=False,
            timeout=120   # ✅ prevent hanging
        )

        data = response.json().get("value", [])

    except Exception as e:
        print(f"❌ API error: {e}")
        break

    if not data:
        break

    final_rows = []

    for row in data:
        course = row.copy()

        rating = course.pop("Rating", {})
        publishing = course.pop("Publishing", None)

        new_row = {}

        # ✅ Course fields
        for k, v in course.items():
            new_row[k] = v

        # ✅ Rating expand
        if isinstance(rating, dict):
            for k, v in rating.items():
                new_row[f"Rating_{k}"] = v

        # ✅ Publishing expand
        if isinstance(publishing, dict):
            # ✅ already structured
            for k, v in publishing.items():
                new_row[f"Publishing_{k}"] = v

        elif isinstance(publishing, str):
            # ✅ it's a link → fetch details (slow part)
            try:
                pub_resp = requests.get(
                    publishing,
                    auth=(username, password),
                    verify=False,
                    timeout=30
                )

                pub_data = pub_resp.json()

                if isinstance(pub_data, dict):
                    for k, v in pub_data.items():
                        new_row[f"Publishing_{k}"] = v

            except:
                # ✅ silently skip if fails
                pass

        # ✅ append row
        final_rows.append(new_row)

    df = pd.DataFrame(final_rows)

    # ✅ Clean data
    df = df.astype(str)
    for col in df.columns:
        df[col] = df[col].str[:4000]

    # ✅ Create table first time
    if first_batch:
        cols = [f"[{col}] NVARCHAR(MAX)" for col in df.columns]

        cursor.execute(f"""
        IF OBJECT_ID('{table_name}', 'U') IS NOT NULL
            DROP TABLE {table_name};

        CREATE TABLE {table_name} (
            {', '.join(cols)}
        )
        """)
        conn.commit()
        first_batch = False

    # ✅ Insert data
    cols = ",".join([f"[{c}]" for c in df.columns])
    placeholders = ",".join(["?"] * len(df.columns))

    cursor.executemany(
        f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})",
        df.values.tolist()
    )
    conn.commit()

    skip += top
    print(f"Inserted {len(df)} rows... skip={skip}")

print("✅ Courses with Rating + Publishing loaded")

# Loading 'CourseCategories' table with Courses expand

In [ ]:
import requests
import pandas as pd
import pyodbc
import urllib3
import warnings
warnings.filterwarnings("ignore")

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- Credentials ---
username = "6b3661eb-7413-444c-9ee7-648d0a0915e6"
password = "6b3661eb-7413-444c-9ee7-648d0a0915e6"

# --- SQL Connection ---
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=BRW03WFTATHASD;"
    "DATABASE=WFTD_data_warehouse;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()
cursor.fast_executemany = True

# --- API (expand) ---
base_url = "https://ne-api.365.systems/odata/v2/CourseCategories?$expand=Courses,ParentCategory"

top = 1000
skip = 0
first_batch = True

table_name = "CourseCategories"

while True:
    url = f"{base_url}&$top={top}&$skip={skip}"

    response = requests.get(
        url,
        auth=(username, password),
        verify=False
    )

    data = response.json().get("value", [])

    if not data:
        break

    final_data = []

    for row in data:
        cat = row.copy()
        courses = cat.pop("Courses", [])

        # If no courses → still keep category
        if not courses:
            final_data.append(cat)

        for c in courses:
            merged = {}

            # category columns
            for k, v in cat.items():
                merged[f"Category_{k}"] = v

            # course columns
            for k, v in c.items():
                merged[f"Course_{k}"] = v

            final_data.append(merged)

    df = pd.DataFrame(final_data)

    # ✅ Clean data
    df = df.astype(str).applymap(lambda x: x[:4000])

    # ✅ Create table
    if first_batch:
        cols = [f"[{col}] NVARCHAR(MAX)" for col in df.columns]

        cursor.execute(f"""
        IF OBJECT_ID('{table_name}', 'U') IS NOT NULL
            DROP TABLE {table_name};

        CREATE TABLE {table_name} (
            {', '.join(cols)}
        )
        """)
        conn.commit()
        first_batch = False

    # ✅ Insert
    cols = ",".join([f"[{c}]" for c in df.columns])
    placeholders = ",".join(["?"] * len(df.columns))

    cursor.executemany(
        f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})",
        df.values.tolist()
    )
    conn.commit()

    skip += top
    print(f"Inserted batch... skip={skip}")

print("✅ CourseCategories table loaded successfully")

# Load 'Users' Table

In [ ]:
import requests
import pandas as pd
import pyodbc
import urllib3
import warnings
warnings.filterwarnings("ignore")

# suppress SSL warning
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- Credentials ---
username = "6b3661eb-7413-444c-9ee7-648d0a0915e6"
password = "6b3661eb-7413-444c-9ee7-648d0a0915e6"

# --- SQL Connection ---
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=BRW03WFTATHASD;"
    "DATABASE=WFTD_data_warehouse;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()
cursor.fast_executemany = True

# --- API ---
base_url = "https://api.365.systems/odata/v2/Users"
table_name = "Users"

top = 5000
skip = 0
first_batch = True

while True:
    url = f"{base_url}?$top={top}&$skip={skip}"

    response = requests.get(
        url,
        auth=(username, password),
        verify=False
    )

    data = response.json().get("value", [])

    if not data:
        break

    df = pd.DataFrame(data)

    # ✅ Clean data (fix for long text)
    df = df.astype(str).applymap(lambda x: x[:4000])

    # ✅ Create table
    if first_batch:
        cols = [f"[{col}] NVARCHAR(MAX)" for col in df.columns]

        cursor.execute(f"""
        IF OBJECT_ID('{table_name}', 'U') IS NOT NULL
            DROP TABLE {table_name};

        CREATE TABLE {table_name} (
            {', '.join(cols)}
        )
        """)
        conn.commit()
        first_batch = False

    # ✅ Insert data
    if not df.empty:
        cols = ",".join([f"[{c}]" for c in df.columns])
        placeholders = ",".join(["?"] * len(df.columns))

        cursor.executemany(
            f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})",
            df.values.tolist()
        )
        conn.commit()

    skip += top
    print(f"Inserted {len(df)} rows... skip={skip}")

print("✅ Users table loaded successfully")

# Load 'CourseSessions' Table

In [ ]:
import requests
import pandas as pd
import pyodbc
import urllib3
import warnings
warnings.filterwarnings("ignore")

# suppress SSL warning
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- Credentials ---
username = "6b3661eb-7413-444c-9ee7-648d0a0915e6"
password = "6b3661eb-7413-444c-9ee7-648d0a0915e6"

# --- SQL Connection ---
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=BRW03WFTATHASD;"
    "DATABASE=WFTD_data_warehouse;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()
cursor.fast_executemany = True

# --- API ---
base_url = "https://api.365.systems/odata/v2/CourseSessions"
table_name = "CourseSessions"

top = 5000
skip = 0
first_batch = True

while True:
    url = f"{base_url}?$top={top}&$skip={skip}"

    response = requests.get(
        url,
        auth=(username, password),
        verify=False
    )

    data = response.json().get("value", [])

    if not data:
        break

    df = pd.DataFrame(data)

    # ✅ Clean data (important)
    df = df.astype(str).applymap(lambda x: x[:4000])

    # ✅ Create table
    if first_batch:
        cols = [f"[{col}] NVARCHAR(MAX)" for col in df.columns]

        cursor.execute(f"""
        IF OBJECT_ID('{table_name}', 'U') IS NOT NULL
            DROP TABLE {table_name};

        CREATE TABLE {table_name} (
            {', '.join(cols)}
        )
        """)
        conn.commit()
        first_batch = False

    # ✅ Insert data
    if not df.empty:
        cols = ",".join([f"[{c}]" for c in df.columns])
        placeholders = ",".join(["?"] * len(df.columns))

        cursor.executemany(
            f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})",
            df.values.tolist()
        )
        conn.commit()

    skip += top
    print(f"Inserted {len(df)} rows... skip={skip}")

print("✅ CourseSessions table loaded successfully")

# Load 'TrainingPlans' Table

In [ ]:
import requests
import pandas as pd
import pyodbc
import urllib3
import warnings

warnings.filterwarnings("ignore")
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- Credentials ---
username = "6b3661eb-7413-444c-9ee7-648d0a0915e6"
password = "6b3661eb-7413-444c-9ee7-648d0a0915e6"

# --- SQL Connection ---
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=BRW03WFTATHASD;"
    "DATABASE=WFTD_data_warehouse;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()
cursor.fast_executemany = True

# ✅ EXPAND COURSES
base_url = "https://api.365.systems/odata/v2/TrainingPlans?$expand=Courses"

table_name = "TrainingPlans"

top = 1000
skip = 0
first_batch = True

while True:
    url = f"{base_url}&$top={top}&$skip={skip}"

    response = requests.get(
        url,
        auth=(username, password),
        verify=False
    )

    data = response.json().get("value", [])

    if not data:
        break

    final_rows = []

    for row in data:
        plan = row.copy()
        courses = plan.pop("Courses", [])

        # flatten
        for c in courses:
            new_row = {}

            # TrainingPlan fields
            new_row["TrainingPlanId"] = plan.get("Id")
            new_row["TrainingPlanTitle"] = plan.get("Title")
            new_row["CourseCatalogId"] = plan.get("CourseCatalogId")

            new_row["TrainingPlanCreatedAt"] = plan.get("CreatedAt")
            new_row["TrainingPlanModifiedAt"] = plan.get("ModifiedAt")

            # Course fields
            for k, v in c.items():
                new_row[f"Course_{k}"] = v

            final_rows.append(new_row)

    if not final_rows:
        skip += top
        continue

    df = pd.DataFrame(final_rows)

    # ✅ Clean data
    df = df.astype(str)
    for col in df.columns:
        df[col] = df[col].str[:4000]

    # ✅ Create table
    if first_batch:
        cols = [f"[{col}] NVARCHAR(MAX)" for col in df.columns]

        cursor.execute(f"""
        IF OBJECT_ID('{table_name}', 'U') IS NOT NULL
            DROP TABLE {table_name};

        CREATE TABLE {table_name} (
            {', '.join(cols)}
        )
        """)
        conn.commit()
        first_batch = False

    # ✅ Insert
    cols = ",".join([f"[{c}]" for c in df.columns])
    placeholders = ",".join(["?"] * len(df.columns))

    cursor.executemany(
        f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})",
        df.values.tolist()
    )
    conn.commit()

    skip += top
    print(f"Inserted {len(df)} rows... skip={skip}")

print("✅ TrainingPlans loaded")

# Load 'CourseCatalogs' Table

In [ ]:
import requests
import pandas as pd
import pyodbc
import urllib3
import warnings
warnings.filterwarnings("ignore")

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

username = "6b3661eb-7413-444c-9ee7-648d0a0915e6"
password = "6b3661eb-7413-444c-9ee7-648d0a0915e6"

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=BRW03WFTATHASD;"
    "DATABASE=WFTD_data_warehouse;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()
cursor.fast_executemany = True

base_url = "https://api.365.systems/odata/v2/CourseCatalogs"
table_name = "CourseCatalogs"

top = 5000
skip = 0
first_batch = True

while True:
    url = f"{base_url}?$top={top}&$skip={skip}"

    response = requests.get(
        url,
        auth=(username, password),
        verify=False
    )

    data = response.json().get("value", [])

    if not data:
        break

    df = pd.DataFrame(data)

    df = df.astype(str).applymap(lambda x: x[:4000])

    if first_batch:
        cols = [f"[{col}] NVARCHAR(MAX)" for col in df.columns]

        cursor.execute(f"""
        IF OBJECT_ID('{table_name}', 'U') IS NOT NULL
            DROP TABLE {table_name};

        CREATE TABLE {table_name} (
            {', '.join(cols)}
        )
        """)
        conn.commit()
        first_batch = False

    if not df.empty:
        cols = ",".join([f"[{c}]" for c in df.columns])
        placeholders = ",".join(["?"] * len(df.columns))

        cursor.executemany(
            f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})",
            df.values.tolist()
        )
        conn.commit()

    skip += top
    print(f"Inserted {len(df)} rows... skip={skip}")

print("✅ CourseCatalogs loaded")

# Load 'Attendances' Table. Enrollments -> Expand -> Attendances. 
## 2015-2020

In [ ]:
import requests
import pandas as pd
import pyodbc
import urllib3
import warnings
warnings.filterwarnings("ignore")

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

username = "6b3661eb-7413-444c-9ee7-648d0a0915e6"
password = "6b3661eb-7413-444c-9ee7-648d0a0915e6"

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=BRW03WFTATHASD;"
    "DATABASE=WFTD_data_warehouse;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()
cursor.fast_executemany = True

table_name = "Attendances"
base_url = "https://ne-api.365.systems/odata/v2/Enrollments"

top = 2000
first_batch = True   # ✅ only affects first insert

# ✅ CHANGE RANGE AS NEEDED PER RUN
for year in range(2015, 2021):   # ← change per run

    start = f"{year}-01-01T00:00:00Z"
    end = f"{year+1}-01-01T00:00:00Z"

    print(f"Processing year {year}")

    skip = 0

    while True:
        url = (
            f"{base_url}?"
            f"$filter=RegistrationDate ge {start} and RegistrationDate lt {end}"
            f"&$expand=Attendances"
            f"&$top={top}&$skip={skip}"
        )

        response = requests.get(
            url,
            auth=(username, password),
            verify=False
        )

        data = response.json().get("value", [])

        if not data:
            break

        final_rows = []

        for row in data:
            user = row.get("UserLoginName")
            attendances = row.get("Attendances", [])

            for att in attendances:
                new_row = {"UserLoginName": user}

                for k, v in att.items():
                    new_row[k] = v

                final_rows.append(new_row)

        if not final_rows:
            skip += top
            continue

        df = pd.DataFrame(final_rows)

        # ✅ Clean
        df = df.astype(str).applymap(lambda x: x[:4000])

        # ✅ Create table ONLY if not exists
        if first_batch:
            cols = [f"[{col}] NVARCHAR(MAX)" for col in df.columns]

            cursor.execute(f"""
            IF OBJECT_ID('{table_name}', 'U') IS NULL
            CREATE TABLE {table_name} (
                {', '.join(cols)}
            )
            """)
            conn.commit()
            first_batch = False

        cols = ",".join([f"[{c}]" for c in df.columns])
        placeholders = ",".join(["?"] * len(df.columns))

        cursor.executemany(
            f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})",
            df.values.tolist()
        )
        conn.commit()

        skip += top
        print(f"Year {year} → inserted {len(df)} rows")

print("✅ Run complete")

# Load 'Attendances' Table. Enrollments -> Expand -> Attendances. 
## 2021-2023

In [ ]:
import requests
import pandas as pd
import pyodbc
import urllib3
import warnings
warnings.filterwarnings("ignore")

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

username = "6b3661eb-7413-444c-9ee7-648d0a0915e6"
password = "6b3661eb-7413-444c-9ee7-648d0a0915e6"

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=BRW03WFTATHASD;"
    "DATABASE=WFTD_data_warehouse;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()
cursor.fast_executemany = True

table_name = "Attendances"
base_url = "https://ne-api.365.systems/odata/v2/Enrollments"

top = 2000
first_batch = True   # ✅ only affects first insert

# ✅ CHANGE RANGE AS NEEDED PER RUN
for year in range(2021, 2024):   # ← change per run

    start = f"{year}-01-01T00:00:00Z"
    end = f"{year+1}-01-01T00:00:00Z"

    print(f"Processing year {year}")

    skip = 0

    while True:
        url = (
            f"{base_url}?"
            f"$filter=RegistrationDate ge {start} and RegistrationDate lt {end}"
            f"&$expand=Attendances"
            f"&$top={top}&$skip={skip}"
        )

        response = requests.get(
            url,
            auth=(username, password),
            verify=False
        )

        data = response.json().get("value", [])

        if not data:
            break

        final_rows = []

        for row in data:
            user = row.get("UserLoginName")
            attendances = row.get("Attendances", [])

            for att in attendances:
                new_row = {"UserLoginName": user}

                for k, v in att.items():
                    new_row[k] = v

                final_rows.append(new_row)

        if not final_rows:
            skip += top
            continue

        df = pd.DataFrame(final_rows)

        # ✅ Clean
        df = df.astype(str).applymap(lambda x: x[:4000])

        # ✅ Create table ONLY if not exists
        if first_batch:
            cols = [f"[{col}] NVARCHAR(MAX)" for col in df.columns]

            cursor.execute(f"""
            IF OBJECT_ID('{table_name}', 'U') IS NULL
            CREATE TABLE {table_name} (
                {', '.join(cols)}
            )
            """)
            conn.commit()
            first_batch = False

        cols = ",".join([f"[{c}]" for c in df.columns])
        placeholders = ",".join(["?"] * len(df.columns))

        cursor.executemany(
            f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})",
            df.values.tolist()
        )
        conn.commit()

        skip += top
        print(f"Year {year} → inserted {len(df)} rows")

print("✅ Run complete")

# Load 'Attendances' Table. Enrollments -> Expand -> Attendances. 
## 2024

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import requests
import pandas as pd
import pyodbc
import urllib3
from datetime import datetime, timedelta
import time

# suppress SSL warning
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- Credentials ---
username = "6b3661eb-7413-444c-9ee7-648d0a0915e6"
password = "6b3661eb-7413-444c-9ee7-648d0a0915e6"

# --- SQL Connection ---
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=BRW03WFTATHASD;"
    "DATABASE=WFTD_data_warehouse;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()
cursor.fast_executemany = True

# --- API ---
base_url = "https://ne-api.365.systems/odata/v2/Enrollments"
table_name = "Attendances"

top = 2000
first_batch = True

# ✅ PROCESS ONLY 2024 (you can change later)
for year in range(2024, 2025):

    current_date = datetime(year, 1, 1)

    while current_date.year == year:

        next_month = (current_date.replace(day=28) + timedelta(days=4)).replace(day=1)

        start = current_date.strftime("%Y-%m-%dT00:00:00Z")
        end = next_month.strftime("%Y-%m-%dT00:00:00Z")

        print(f"\nProcessing {start} → {end}")

        skip = 0

        while True:

            url = (
                f"{base_url}?"
                f"$filter=RegistrationDate ge {start} and RegistrationDate lt {end}"
                f"&$expand=Attendances"
                f"&$top={top}&$skip={skip}"
            )

            retry_count = 0
            max_retry = 5

            while retry_count < max_retry:
                try:
                    response = requests.get(
                        url,
                        auth=(username, password),
                        verify=False,
                        timeout=120
                    )

                    response.raise_for_status()
                    break  # success

                except Exception as e:
                    retry_count += 1
                    print(f"Retry {retry_count} due to error: {e}")
                    time.sleep(5)

            if retry_count == max_retry:
                print("Max retries reached, moving to next batch...")
                break

            data = response.json().get("value", [])

            if not data:
                break

            final_rows = []

            for row in data:
                user = row.get("UserLoginName")
                attendances = row.get("Attendances", [])

                for att in attendances:
                    new_row = {"UserLoginName": user}

                    for k, v in att.items():
                        new_row[k] = v

                    final_rows.append(new_row)

            if not final_rows:
                skip += top
                continue

            df = pd.DataFrame(final_rows)

            # ✅ Clean data safely
            df = df.astype(str)
            for col in df.columns:
                df[col] = df[col].str.slice(0, 4000)

            # ✅ Create table only once
            if first_batch:
                cols = [f"[{col}] NVARCHAR(MAX)" for col in df.columns]

                cursor.execute(f"""
                IF OBJECT_ID('{table_name}', 'U') IS NULL
                CREATE TABLE {table_name} (
                    {', '.join(cols)}
                )
                """)
                conn.commit()
                first_batch = False

            # ✅ Insert
            cols = ",".join([f"[{c}]" for c in df.columns])
            placeholders = ",".join(["?"] * len(df.columns))

            cursor.executemany(
                f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})",
                df.values.tolist()
            )
            conn.commit()

            skip += top
            print(f"Inserted {len(df)} rows... skip={skip}")

        current_date = next_month

print("\n✅ 2024 Attendances loaded successfully")

# Load 'Attendances' Table. Enrollments -> Expand -> Attendances. 
## 2025

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import requests
import pandas as pd
import pyodbc
import urllib3
from datetime import datetime, timedelta
import time

# suppress SSL warning
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- Credentials ---
username = "6b3661eb-7413-444c-9ee7-648d0a0915e6"
password = "6b3661eb-7413-444c-9ee7-648d0a0915e6"

# --- SQL Connection ---
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=BRW03WFTATHASD;"
    "DATABASE=WFTD_data_warehouse;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()
cursor.fast_executemany = True

# --- API ---
base_url = "https://ne-api.365.systems/odata/v2/Enrollments"
table_name = "Attendances"

top = 2000
first_batch = True

# ✅ PROCESS ONLY 2025 (you can change later)
for year in range(2025, 2026):

    current_date = datetime(year, 1, 1)

    while current_date.year == year:

        next_month = (current_date.replace(day=28) + timedelta(days=4)).replace(day=1)

        start = current_date.strftime("%Y-%m-%dT00:00:00Z")
        end = next_month.strftime("%Y-%m-%dT00:00:00Z")

        print(f"\nProcessing {start} → {end}")

        skip = 0

        while True:

            url = (
                f"{base_url}?"
                f"$filter=RegistrationDate ge {start} and RegistrationDate lt {end}"
                f"&$expand=Attendances"
                f"&$top={top}&$skip={skip}"
            )

            retry_count = 0
            max_retry = 5

            while retry_count < max_retry:
                try:
                    response = requests.get(
                        url,
                        auth=(username, password),
                        verify=False,
                        timeout=120
                    )

                    response.raise_for_status()
                    break  # success

                except Exception as e:
                    retry_count += 1
                    print(f"Retry {retry_count} due to error: {e}")
                    time.sleep(5)

            if retry_count == max_retry:
                print("Max retries reached, moving to next batch...")
                break

            data = response.json().get("value", [])

            if not data:
                break

            final_rows = []

            for row in data:
                user = row.get("UserLoginName")
                attendances = row.get("Attendances", [])

                for att in attendances:
                    new_row = {"UserLoginName": user}

                    for k, v in att.items():
                        new_row[k] = v

                    final_rows.append(new_row)

            if not final_rows:
                skip += top
                continue

            df = pd.DataFrame(final_rows)

            # ✅ Clean data safely
            df = df.astype(str)
            for col in df.columns:
                df[col] = df[col].str.slice(0, 4000)

            # ✅ Create table only once
            if first_batch:
                cols = [f"[{col}] NVARCHAR(MAX)" for col in df.columns]

                cursor.execute(f"""
                IF OBJECT_ID('{table_name}', 'U') IS NULL
                CREATE TABLE {table_name} (
                    {', '.join(cols)}
                )
                """)
                conn.commit()
                first_batch = False

            # ✅ Insert
            cols = ",".join([f"[{c}]" for c in df.columns])
            placeholders = ",".join(["?"] * len(df.columns))

            cursor.executemany(
                f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})",
                df.values.tolist()
            )
            conn.commit()

            skip += top
            print(f"Inserted {len(df)} rows... skip={skip}")

        current_date = next_month

print("\n✅ 2025 Attendances loaded successfully")

# Load 'Attendances' Table. Enrollments -> Expand -> Attendances. 
## 2026

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import requests
import pandas as pd
import pyodbc
import urllib3
from datetime import datetime, timedelta
import time

# suppress SSL warning
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- Credentials ---
username = "6b3661eb-7413-444c-9ee7-648d0a0915e6"
password = "6b3661eb-7413-444c-9ee7-648d0a0915e6"

# --- SQL Connection ---
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=BRW03WFTATHASD;"
    "DATABASE=WFTD_data_warehouse;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()
cursor.fast_executemany = True

# --- API ---
base_url = "https://ne-api.365.systems/odata/v2/Enrollments"
table_name = "Attendances"

top = 2000
first_batch = True

# ✅ PROCESS ONLY 2026 (you can change later)
for year in range(2026, 2027):

    current_date = datetime(year, 1, 1)

    while current_date.year == year:

        next_month = (current_date.replace(day=28) + timedelta(days=4)).replace(day=1)

        start = current_date.strftime("%Y-%m-%dT00:00:00Z")
        end = next_month.strftime("%Y-%m-%dT00:00:00Z")

        print(f"\nProcessing {start} → {end}")

        skip = 0

        while True:

            url = (
                f"{base_url}?"
                f"$filter=RegistrationDate ge {start} and RegistrationDate lt {end}"
                f"&$expand=Attendances"
                f"&$top={top}&$skip={skip}"
            )

            retry_count = 0
            max_retry = 5

            while retry_count < max_retry:
                try:
                    response = requests.get(
                        url,
                        auth=(username, password),
                        verify=False,
                        timeout=120
                    )

                    response.raise_for_status()
                    break  # success

                except Exception as e:
                    retry_count += 1
                    print(f"Retry {retry_count} due to error: {e}")
                    time.sleep(5)

            if retry_count == max_retry:
                print("Max retries reached, moving to next batch...")
                break

            data = response.json().get("value", [])

            if not data:
                break

            final_rows = []

            for row in data:
                user = row.get("UserLoginName")
                attendances = row.get("Attendances", [])

                for att in attendances:
                    new_row = {"UserLoginName": user}

                    for k, v in att.items():
                        new_row[k] = v

                    final_rows.append(new_row)

            if not final_rows:
                skip += top
                continue

            df = pd.DataFrame(final_rows)

            # ✅ Clean data safely
            df = df.astype(str)
            for col in df.columns:
                df[col] = df[col].str.slice(0, 4000)

            # ✅ Create table only once
            if first_batch:
                cols = [f"[{col}] NVARCHAR(MAX)" for col in df.columns]

                cursor.execute(f"""
                IF OBJECT_ID('{table_name}', 'U') IS NULL
                CREATE TABLE {table_name} (
                    {', '.join(cols)}
                )
                """)
                conn.commit()
                first_batch = False

            # ✅ Insert
            cols = ",".join([f"[{c}]" for c in df.columns])
            placeholders = ",".join(["?"] * len(df.columns))

            cursor.executemany(
                f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})",
                df.values.tolist()
            )
            conn.commit()

            skip += top
            print(f"Inserted {len(df)} rows... skip={skip}")

        current_date = next_month

print("\n✅ 2026 Attendances loaded successfully")